# Baseline Modeling (9개 모델 비교)

데이터 로드 → 전처리(클리닝/이상치/집계/스케일링) → 9개 모델 Baseline 비교 → Feature Importance

**목적**: 다양한 모델을 기본 파라미터로 돌려 RMSE를 한눈에 비교하고, 유망 모델을 선별한다.

**모델**: LightGBM, XGBoost, RandomForest, ExtraTrees, Ridge, Lasso, ElasticNet, TabNet, FT-Transformer

## 1. 환경 설정 및 데이터 로드

In [1]:
import os, sys

try:
    import google.colab
    if not os.path.exists("/content/project/setup.py"):
        os.system("pip install -q gdown")
        os.system("gdown --id 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip")
        os.system("unzip -qo /content/code.zip -d /content/project")
        os.makedirs("/content/project/0_data", exist_ok=True)
        os.system("gdown --id 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip")
        os.system("unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data")
        os.remove("/content/project/0_data/dataset.zip")
    if not os.path.exists("/content/project/2_preprocessing/cleaning.py"):
        os.system("gdown --id 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip")
        os.system("unzip -qo /content/preprocessing.zip -d /content/project")
    sys.path.insert(0, "/content/project")
    %run /content/project/setup.py
except ImportError:
    %run ../setup.py

# ─── 이 노트북에서 사용하는 import ───
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings("ignore")

# 트리 모델
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

# 선형 모델
from sklearn.linear_model import Ridge, Lasso, ElasticNet

# 딥러닝 (TabNet, FT-Transformer)
from pytorch_tabnet.tab_model import TabNetRegressor
from rtdl_revisiting_models import FTTransformer

# 스케일링 / 평가 / CV
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.base import clone

# 프로젝트 utils
from utils.config import PROJECT_ROOT, SEED, TARGET_COL
from utils.data import load_all, get_feat_cols, split_xs
from utils.aggregate import merge_with_target
from utils.evaluate import evaluate, postprocess, compare_models

# 전처리 모듈
sys.path.insert(0, os.path.join(PROJECT_ROOT, "2_preprocessing"))
from cleaning import run_cleaning
from outlier import run_outlier_treatment
from aggregation import run_aggregation

# GPU 감지
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HAS_GPU = DEVICE == "cuda"
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if HAS_GPU else ""))

# 데이터 로드
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
ys_train = ys["train"]
ys_val = ys["validation"]

setup 완료
Device: cuda (NVIDIA T600 Laptop GPU)
Xs: (174980, 1091)  |  Ys: train=26,247, val=8,749, test=8,749


## 2. 전처리 파이프라인 (클리닝 → 이상치 → 집계)

In [2]:
# --- Step 1: 클리닝 (상수/고결측/중복 제거 + 결측 imputation) ---
xs_train, xs_val, xs_test, clean_cols, clean_report = run_cleaning(
    xs, feat_cols, xs_dict,
    const_threshold=1e-6,      # std가 이 값 이하인 feature 제거
    missing_threshold=0.5,     # 결측률이 이 비율 이상인 feature 제거
    remove_duplicates=True,    # 값이 완전히 동일한 중복 컬럼 제거
)

클리닝 파이프라인 시작
원본 feature 수: 1087
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 982개
    컬럼: 1087 → 982 (105개 제거)
    DataFrame: (104988, 986)

[고결측 제거] threshold=50%
  제거: 5개, 잔여: 977개
    컬럼: 982 → 977 (5개 제거)
    DataFrame: (104988, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 26개, 잔여: 951개
    컬럼: 977 → 951 (26개 제거)
    DataFrame: (104988, 955)

[결측 imputation] train median 기준
  imputation 후 잔여 결측: 0

클리닝 완료: 1087 → 951 features (136개 제거)
  train: (104988, 955)
  val:   (34996, 955)
  test:  (34996, 955)


In [3]:
# --- Step 2: 이상치 처리 (Winsorization) ---
xs_train, xs_val, xs_test, outlier_report = run_outlier_treatment(
    xs_train, xs_val, xs_test, clean_cols,
    lower_pct=0.01,   # 하위 분위수 경계 (이 미만 값을 클리핑)
    upper_pct=0.99,   # 상위 분위수 경계 (이 초과 값을 클리핑)
)

이상치 처리 파이프라인 시작
[이상치 탐지] IQR × 1.5
  이상치 > 5%: 165개
  이상치 > 10%: 63개
[Winsorization] lower=1%, upper=99%
  적용 feature: 951개
[이상치 탐지] IQR × 1.5
  이상치 > 5%: 164개
  이상치 > 10%: 63개

이상치 처리 완료
  이상치 >5%  feature: 165 → 164 (1개 감소)
  이상치 >10% feature: 63 → 63 (0개 감소)
  train: (104988, 955)


In [4]:
# --- Step 3: Die → Unit 집계 ---
unit_train, unit_val, unit_test, unit_feat_cols = run_aggregation(
    xs_train, xs_val, xs_test, clean_cols,
    agg_funcs=["mean", "std", "min", "max"],  # die 4개를 unit으로 집계할 통계량
    use_position_pivot=False,                   # True면 position별 피벗 feature도 추가
    save_csv=False,                             # baseline에서는 CSV 저장 안 함
)

Die → Unit 집계 시작
  agg_funcs: ['mean', 'std', 'min', 'max']
  position_pivot: False
집계 완료: 26,247 units × 3,804 features (agg: ['mean', 'std', 'min', 'max'])
집계 완료: 8,749 units × 3,804 features (agg: ['mean', 'std', 'min', 'max'])
집계 완료: 8,749 units × 3,804 features (agg: ['mean', 'std', 'min', 'max'])

집계 결과:
  train: (26247, 3804)
  val:   (8749, 3804)
  test:  (8749, 3804)
  feature 수: 3804


## 3. Target Merge 및 학습 데이터 준비

In [5]:
# Target merge (train / val / test)
X_train, y_train = merge_with_target(unit_train, split="train")
X_val, y_val = merge_with_target(unit_val, split="validation")
X_test, y_test = merge_with_target(unit_test, split="test")

# NaN 확인
assert X_train.isnull().sum().sum() == 0, "X_train에 NaN 존재!"
assert X_val.isnull().sum().sum() == 0, "X_val에 NaN 존재!"
assert X_test.isnull().sum().sum() == 0, "X_test에 NaN 존재!"

# RobustScaler (선형/딥러닝 모델용이지만, 트리 모델에 영향 없으므로 1벌로 통일)
scaler = RobustScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_s = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"\ny_train: mean={y_train.mean():.6f}, zero={(y_train==0).mean()*100:.1f}%")
print(f"y_val:   mean={y_val.mean():.6f}, zero={(y_val==0).mean()*100:.1f}%")
print(f"y_test:  mean={y_test.mean():.6f}, zero={(y_test==0).mean()*100:.1f}%")

# 단순 baseline (비교 기준선)
print(f"\n--- 단순 Baseline ---")
evaluate(y_val, np.zeros(len(y_val)), label="all-zero")
evaluate(y_val, np.full(len(y_val), y_train.mean()), label="train-mean")

Merge (train): X=(26247, 3804), y=(26247,), y_mean=0.002515
Merge (validation): X=(8749, 3804), y=(8749,), y_mean=0.002505
Merge (test): X=(8749, 3804), y=(8749,), y_mean=0.002610

y_train: mean=0.002515, zero=70.8%
y_val:   mean=0.002505, zero=70.8%
y_test:  mean=0.002610, zero=70.8%

--- 단순 Baseline ---
[all-zero] RMSE = 0.006331  (n=8,749, zero=6,194(70.8%))
[train-mean] RMSE = 0.005814  (n=8,749, zero=6,194(70.8%))


0.00581430473855135

In [6]:
# 임시 --------------------------------------------------------------------------
# ===== 1/10 샘플링 (빠른 실험용) =====
SAMPLE_FRAC = 0.1
sample_idx = X_train_s.sample(frac=SAMPLE_FRAC, random_state=SEED).index
X_train_s = X_train_s.loc[sample_idx].reset_index(drop=True)
y_train = y_train.loc[sample_idx].reset_index(drop=True)
print(f"샘플링: {len(sample_idx)} → {len(X_train_s)} ({SAMPLE_FRAC*100:.0f}%)")

샘플링: 2625 → 2625 (10%)


In [9]:
print(X_train_s.shape)

(2625, 3804)


## 4. 9개 모델 Baseline 비교

트리 4종 + 선형 3종 + 딥러닝 2종을 **기본 파라미터**로 학습하여 Val/Test RMSE를 비교

In [7]:
# ─── 9개 모델 정의 (기본 파라미터) ───
models = {
    "LightGBM": lgb.LGBMRegressor(
        n_estimators=1000, random_state=SEED, n_jobs=-1, verbose=-1,
        device="gpu" if HAS_GPU else "cpu",
    ),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=1000, random_state=SEED, n_jobs=-1, verbosity=0,
        early_stopping_rounds=50,
        device=DEVICE,
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=500, random_state=SEED, n_jobs=-1,
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=500, random_state=SEED, n_jobs=-1,
    ),
    "Ridge": Ridge(alpha=1.0, random_state=SEED),
    "Lasso": Lasso(alpha=0.0001, random_state=SEED, max_iter=5000),
    "ElasticNet": ElasticNet(alpha=0.0001, l1_ratio=0.5, random_state=SEED, max_iter=5000),
}

In [8]:
# ─── K-Fold CV 함수 (sklearn 모델용) ───
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def run_kfold_cv(name, model, X, y):
    """sklearn 모델의 K-Fold CV RMSE를 반환"""
    fold_scores = []
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
        m = clone(model)
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        if name == "LightGBM":
            m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        elif name == "XGBoost":
            m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        else:
            m.fit(X_tr, y_tr)

        pred = postprocess(m.predict(X_va))
        fold_scores.append(np.sqrt(mean_squared_error(y_va, pred)))
    return fold_scores

# ─── sklearn 기반 7개 모델: K-Fold CV + Val + Test ───
results = {}
trained_models = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"[{name}]")

    # K-Fold CV
    cv_scores = run_kfold_cv(name, model, X_train_s, y_train)
    print(f"  CV RMSE: {np.mean(cv_scores):.6f} (+/- {np.std(cv_scores):.6f})")

    # 전체 train으로 재학습 → Val/Test 예측
    if name == "LightGBM":
        model.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    elif name == "XGBoost":
        model.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=False)
    else:
        model.fit(X_train_s, y_train)

    pred_val = postprocess(model.predict(X_val_s))
    pred_test = postprocess(model.predict(X_test_s))
    val_rmse = evaluate(y_val, pred_val, label=f"{name} (val)")
    test_rmse = evaluate(y_test, pred_test, label=f"{name} (test)")

    results[name] = {
        "cv_scores": cv_scores, "cv_mean": np.mean(cv_scores), "cv_std": np.std(cv_scores),
        "val_pred": pred_val, "test_pred": pred_test,
        "val_rmse": val_rmse, "test_rmse": test_rmse,
    }
    trained_models[name] = model

# ─── TabNet (K-Fold CV + Val + Test) ───
print(f"\n{'='*50}")
print(f"[TabNet]")

tabnet_cv_scores = []
for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train_s)):
    tn = TabNetRegressor(seed=SEED, verbose=0, device_name=DEVICE,
                         optimizer_params=dict(lr=2e-2),
                         scheduler_params={"step_size": 50, "gamma": 0.9},
                         scheduler_fn=torch.optim.lr_scheduler.StepLR)
    tn.fit(X_train_s.values[tr_idx], y_train.values[tr_idx].reshape(-1, 1),
           eval_set=[(X_train_s.values[va_idx], y_train.values[va_idx].reshape(-1, 1))],
           max_epochs=200, patience=30, batch_size=1024, eval_metric=["rmse"])
    pred_fold = postprocess(tn.predict(X_train_s.values[va_idx]).flatten())
    tabnet_cv_scores.append(np.sqrt(mean_squared_error(y_train.values[va_idx], pred_fold)))
print(f"  CV RMSE: {np.mean(tabnet_cv_scores):.6f} (+/- {np.std(tabnet_cv_scores):.6f})")

tabnet = TabNetRegressor(seed=SEED, verbose=0, device_name=DEVICE,
                         optimizer_params=dict(lr=2e-2),
                         scheduler_params={"step_size": 50, "gamma": 0.9},
                         scheduler_fn=torch.optim.lr_scheduler.StepLR)
tabnet.fit(X_train_s.values, y_train.values.reshape(-1, 1),
           eval_set=[(X_val_s.values, y_val.values.reshape(-1, 1))],
           max_epochs=200, patience=30, batch_size=1024, eval_metric=["rmse"])
pred_val_tabnet = postprocess(tabnet.predict(X_val_s.values).flatten())
pred_test_tabnet = postprocess(tabnet.predict(X_test_s.values).flatten())
results["TabNet"] = {
    "cv_scores": tabnet_cv_scores, "cv_mean": np.mean(tabnet_cv_scores), "cv_std": np.std(tabnet_cv_scores),
    "val_pred": pred_val_tabnet, "test_pred": pred_test_tabnet,
    "val_rmse": evaluate(y_val, pred_val_tabnet, label="TabNet (val)"),
    "test_rmse": evaluate(y_test, pred_test_tabnet, label="TabNet (test)"),
}
trained_models["TabNet"] = tabnet

# ─── FT-Transformer (K-Fold CV + Val + Test) ───
print(f"\n{'='*50}")
print(f"[FT-Transformer]")
n_features = X_train_s.shape[1]

def train_ft_transformer(X_tr, y_tr, X_va, y_va, max_epochs=200, patience=30):
    """FT-Transformer 학습 후 val 예측값 반환"""
    ft_kwargs = FTTransformer.get_default_kwargs(n_blocks=3)
    ft_kwargs["d_out"] = 1
    m = FTTransformer(n_cont_features=X_tr.shape[1], cat_cardinalities=[], **ft_kwargs).to(DEVICE)
    opt = torch.optim.AdamW(m.make_parameter_groups(), lr=1e-4, weight_decay=1e-5)

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32).reshape(-1, 1).to(DEVICE)
    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(DEVICE)

    best_rmse, counter, best_state = float("inf"), 0, None
    for ep in range(max_epochs):
        m.train()
        idx = torch.randperm(len(X_tr_t), device=DEVICE)
        for s in range(0, len(X_tr_t), 1024):
            bi = idx[s:s+1024]
            loss = torch.nn.functional.mse_loss(m(X_tr_t[bi], None), y_tr_t[bi])
            opt.zero_grad(); loss.backward(); opt.step()

        m.eval()
        with torch.no_grad():
            pv = m(X_va_t, None).cpu().numpy().flatten()
        rmse_ep = np.sqrt(mean_squared_error(y_va, np.clip(pv, 0, None)))
        if rmse_ep < best_rmse:
            best_rmse = rmse_ep
            best_state = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

    m.load_state_dict(best_state)
    m.to(DEVICE).eval()
    return m

ft_cv_scores = []
for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train_s)):
    m = train_ft_transformer(X_train_s.values[tr_idx], y_train.values[tr_idx],
                             X_train_s.values[va_idx], y_train.values[va_idx])
    with torch.no_grad():
        pv = postprocess(m(torch.tensor(X_train_s.values[va_idx], dtype=torch.float32).to(DEVICE), None).cpu().numpy().flatten())
    ft_cv_scores.append(np.sqrt(mean_squared_error(y_train.values[va_idx], pv)))
print(f"  CV RMSE: {np.mean(ft_cv_scores):.6f} (+/- {np.std(ft_cv_scores):.6f})")

ft_model = train_ft_transformer(X_train_s.values, y_train.values, X_val_s.values, y_val.values)
with torch.no_grad():
    pred_val_ftt = postprocess(ft_model(torch.tensor(X_val_s.values, dtype=torch.float32).to(DEVICE), None).cpu().numpy().flatten())
    pred_test_ftt = postprocess(ft_model(torch.tensor(X_test_s.values, dtype=torch.float32).to(DEVICE), None).cpu().numpy().flatten())
results["FT-Transformer"] = {
    "cv_scores": ft_cv_scores, "cv_mean": np.mean(ft_cv_scores), "cv_std": np.std(ft_cv_scores),
    "val_pred": pred_val_ftt, "test_pred": pred_test_ftt,
    "val_rmse": evaluate(y_val, pred_val_ftt, label="FT-Transformer (val)"),
    "test_rmse": evaluate(y_test, pred_test_ftt, label="FT-Transformer (test)"),
}
trained_models["FT-Transformer"] = ft_model

print(f"\n{'='*50}")
print(f"{len(results)}개 모델 학습 완료")


[LightGBM]
  CV RMSE: 0.005550 (+/- 0.000610)
[LightGBM (val)] RMSE = 0.005819  (n=8,749, zero=6,194(70.8%))
[LightGBM (test)] RMSE = 0.008490  (n=8,749, zero=6,194(70.8%))

[XGBoost]
  CV RMSE: 0.005576 (+/- 0.000598)
[XGBoost (val)] RMSE = 0.005834  (n=8,749, zero=6,194(70.8%))
[XGBoost (test)] RMSE = 0.008514  (n=8,749, zero=6,194(70.8%))

[RandomForest]


KeyboardInterrupt: 

In [ ]:
# ─── 9개 모델 RMSE 비교표 (CV / Val / Test) ───
comparison = pd.DataFrame([
    {
        "Model": name,
        "CV RMSE (mean)": r["cv_mean"],
        "CV RMSE (std)": r["cv_std"],
        "Val RMSE": r["val_rmse"],
        "Test RMSE": r["test_rmse"],
    }
    for name, r in results.items()
]).sort_values("Val RMSE").reset_index(drop=True)
comparison.index += 1

comparison

## 5. Feature Importance (트리 모델 4종 비교)

LightGBM, XGBoost, RandomForest, ExtraTrees의 feature importance를 비교하여 공통 핵심 피처를 파악

In [ ]:
# ─── 트리 모델 4종 Feature Importance 추출 ───
tree_models = ["LightGBM", "XGBoost", "RandomForest", "ExtraTrees"]
importance_dict = {}

for name in tree_models:
    model = trained_models[name]
    imp = model.feature_importances_
    df = pd.DataFrame({"feature": X_train_s.columns, "importance": imp})
    # 정규화 (모델 간 스케일 통일)
    df["importance_norm"] = df["importance"] / df["importance"].sum()
    df = df.sort_values("importance", ascending=False).reset_index(drop=True)
    df.index += 1
    # 원본 feature명 추출 (X123_mean → X123)
    df["original_feature"] = df["feature"].str.replace(r"_(mean|std|min|max|range|median|skew)$", "", regex=True)
    df["agg_type"] = df["feature"].str.extract(r"_(mean|std|min|max|range|median|skew)$")[0]
    importance_dict[name] = df

# 모델별 Top 10 출력
for name in tree_models:
    df = importance_dict[name]
    print(f"\n{'='*50}")
    print(f"[{name}] Feature Importance Top 10")
    print(df[["feature", "importance", "importance_norm"]].head(10).to_string())
    zero_imp = (df["importance"] == 0).sum()
    print(f"중요도 = 0: {zero_imp}개 / {len(df)}개 ({zero_imp/len(df)*100:.1f}%)")

In [ ]:
# ─── 4개 모델 공통 중요 feature 분석 (원본 X번호 기준) ───
# 각 모델에서 원본 feature별 정규화 중요도 합산 → 4개 모델 평균
orig_imp_per_model = {}
for name in tree_models:
    df = importance_dict[name]
    orig = df.groupby("original_feature")["importance_norm"].sum()
    orig_imp_per_model[name] = orig

orig_imp_all = pd.DataFrame(orig_imp_per_model)
orig_imp_all["mean_importance"] = orig_imp_all.mean(axis=1)
orig_imp_all = orig_imp_all.sort_values("mean_importance", ascending=False)

print("=" * 70)
print("원본 Feature 기준 중요도 Top 20 (4개 트리 모델 평균)")
print("=" * 70)
print(orig_imp_all.head(20).round(6).to_string())

# 시각화: 모델별 Top 20 비교
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 좌: 4개 모델 평균 Top 20
top20 = orig_imp_all.head(20)
axes[0].barh(range(len(top20)), top20["mean_importance"], color="steelblue")
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels(top20.index, fontsize=9)
axes[0].invert_yaxis()
axes[0].set_title("원본 Feature 중요도 Top 20 (4모델 평균)")
axes[0].set_xlabel("Normalized Importance (mean)")

# 우: 모델별 Top 20 히트맵
top20_models = orig_imp_all.head(20)[tree_models]
import seaborn as sns
sns.heatmap(top20_models, annot=True, fmt=".4f", cmap="YlOrRd",
            ax=axes[1], xticklabels=True, yticklabels=True)
axes[1].set_title("모델별 Feature 중요도 비교 (Top 20)")

plt.tight_layout()
plt.show()

## 6. 예측 분포 확인

In [ ]:
# ─── 최고 모델의 예측 분포 확인 ───
best_model_name = comparison.iloc[0]["Model"]
best_pred_val = results[best_model_name]["val_pred"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 실제 vs 예측 히스토그램
axes[0].hist(y_val, bins=50, alpha=0.7, label="actual", color="steelblue")
axes[0].hist(best_pred_val, bins=50, alpha=0.7, label="predicted", color="coral")
axes[0].set_title(f"Actual vs Predicted ({best_model_name})")
axes[0].legend()
axes[0].set_xlabel(TARGET_COL)

# 예측값 분포 확대
axes[1].hist(best_pred_val, bins=50, color="coral", edgecolor="black")
axes[1].set_title(f"Predicted Distribution ({best_model_name})")
axes[1].set_xlabel(TARGET_COL)

# Scatter
axes[2].scatter(y_val, best_pred_val, alpha=0.3, s=5, color="steelblue")
axes[2].plot([0, y_val.max()], [0, y_val.max()], "r--", linewidth=1)
axes[2].set_xlabel("Actual")
axes[2].set_ylabel("Predicted")
axes[2].set_title(f"Actual vs Predicted ({best_model_name})")

plt.tight_layout()
plt.show()

print(f"[{best_model_name}] 예측값 통계:")
print(f"  min={best_pred_val.min():.6f}, max={best_pred_val.max():.6f}, mean={best_pred_val.mean():.6f}")
print(f"  예측 0 비율: {(best_pred_val == 0).mean()*100:.1f}%")